# 06B — Station Coverage / Missing-Data QC

This notebook diagnoses the missing values found after native station extraction.

It answers:
- Which station is responsible for each missing feature?
- Are the 72 missing values from one full station?
- Are gaps caused by `nodata` or `outside` coverage?
- How many complete rows remain for Comb1, Comb1_land, Comb2 and Comb2_land?
- Which features require source-coverage review before modelling?

Run this **after Notebook 06 and before Notebook 07**.


In [1]:

# ============================================================
# 06B — STATION COVERAGE / MISSING-DATA QC
# Khulna Precipitation Downscaling
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

SAMPLES_PATH = PROCESSED_DIR / "station_samples_native_tidy.csv"
EXTRACTION_QC_PATH = PROCESSED_DIR / "station_extraction_qc.csv"
STATIC_QC_PATH = PROCESSED_DIR / "station_static_extraction_qc.csv"

for p in [SAMPLES_PATH, EXTRACTION_QC_PATH, STATIC_QC_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}\n"
            "Run 06_Station_Value_Extraction_NATIVE_FIXED_v2.ipynb first."
        )

print("PROJECT_ROOT =", PROJECT_ROOT)


# ------------------------------------------------------------
# 2. LOAD OUTPUTS FROM NOTEBOOK 06
# ------------------------------------------------------------

samples = pd.read_csv(SAMPLES_PATH)
qc = pd.read_csv(EXTRACTION_QC_PATH)
static_qc = pd.read_csv(STATIC_QC_PATH)

print("\nSamples shape:", samples.shape)
print("Stations:", samples["station_id"].nunique())
print("Years:", sorted(samples["year"].unique()))

display(samples.head())


# ------------------------------------------------------------
# 3. FEATURE LIST
# ------------------------------------------------------------

FEATURES = [
    "CCS",
    "PDIR",
    "GSMaP_MVK",
    "CDR",
    "CHIRPS",
    "IMERG",
    "GSMaP_Gauge",
    "ERA5",
    "DEM",
    "NDVI",
    "LST_Day",
    "Distance_Sea",
]

missing_cols = [c for c in FEATURES if c not in samples.columns]
if missing_cols:
    raise KeyError(
        f"Missing expected feature columns in samples table: {missing_cols}"
    )


# ------------------------------------------------------------
# 4. OVERALL MISSINGNESS BY FEATURE
# ------------------------------------------------------------

overall_missing = pd.DataFrame({
    "missing_count": samples[FEATURES].isna().sum(),
    "missing_pct": samples[FEATURES].isna().mean() * 100.0,
}).sort_values("missing_pct", ascending=False)

print("\n==========================================")
print("OVERALL MISSINGNESS BY FEATURE")
print("==========================================")

display(overall_missing)


# ------------------------------------------------------------
# 5. MISSINGNESS BY STATION × FEATURE
# ------------------------------------------------------------

station_feature_rows = []

for station, g in samples.groupby("station_id"):

    for feature in FEATURES:

        n_total = len(g)
        n_missing = int(g[feature].isna().sum())

        station_feature_rows.append({
            "station_id": station,
            "feature": feature,
            "total_records": n_total,
            "missing_count": n_missing,
            "missing_pct": (100.0 * n_missing / n_total) if n_total else np.nan,
        })

station_feature_missing = pd.DataFrame(station_feature_rows)

print("\n==========================================")
print("MISSINGNESS BY STATION × FEATURE")
print("==========================================")

display(
    station_feature_missing[
        station_feature_missing["missing_count"] > 0
    ].sort_values(
        ["missing_count", "station_id", "feature"],
        ascending=[False, True, True]
    )
)


# ------------------------------------------------------------
# 6. PIVOT TABLE — EASY VIEW
# ------------------------------------------------------------

station_missing_pivot = (
    station_feature_missing
    .pivot(
        index="station_id",
        columns="feature",
        values="missing_count"
    )
    .fillna(0)
    .astype(int)
)

print("\n==========================================")
print("MISSING COUNT PIVOT")
print("==========================================")

display(station_missing_pivot)


# ------------------------------------------------------------
# 7. IDENTIFY FEATURES WHERE ONE STATION IS 100% MISSING
# ------------------------------------------------------------

full_missing_station_feature = station_feature_missing[
    station_feature_missing["missing_pct"] == 100.0
].copy()

print("\n==========================================")
print("100% MISSING STATION–FEATURE PAIRS")
print("==========================================")

if len(full_missing_station_feature):
    display(full_missing_station_feature)
else:
    print("None")


# ------------------------------------------------------------
# 8. STATUS BREAKDOWN BY STATION × FEATURE
# ------------------------------------------------------------

status_by_station = (
    qc.groupby(
        ["station_id", "feature", "status"]
    )
    .size()
    .rename("n")
    .reset_index()
)

print("\n==========================================")
print("EXTRACTION STATUS BY STATION × FEATURE")
print("==========================================")

problem_status = status_by_station[
    status_by_station["status"] != "ok"
].copy()

if len(problem_status):
    display(
        problem_status.sort_values(
            ["station_id", "feature", "status"]
        )
    )
else:
    print("All extraction statuses are OK.")


# ------------------------------------------------------------
# 9. CHECK SPECIFIC SUSPECT FEATURES
# ------------------------------------------------------------

SUSPECT_FEATURES = [
    "CCS",
    "PDIR",
    "CDR",
    "GSMaP_MVK",
    "LST_Day",
    "Distance_Sea",
]

print("\n==========================================")
print("SUSPECT FEATURE DETAILS")
print("==========================================")

for feature in SUSPECT_FEATURES:

    tmp = station_feature_missing[
        station_feature_missing["feature"] == feature
    ].sort_values(
        "missing_count",
        ascending=False
    )

    print(f"\n--- {feature} ---")
    display(tmp)


# ------------------------------------------------------------
# 10. YEAR-WISE MISSINGNESS
# ------------------------------------------------------------

year_feature_rows = []

for year, g in samples.groupby("year"):

    for feature in FEATURES:

        n_total = len(g)
        n_missing = int(g[feature].isna().sum())

        year_feature_rows.append({
            "year": year,
            "feature": feature,
            "total_records": n_total,
            "missing_count": n_missing,
            "missing_pct": (100.0 * n_missing / n_total) if n_total else np.nan,
        })

year_feature_missing = pd.DataFrame(year_feature_rows)

print("\n==========================================")
print("YEAR-WISE MISSINGNESS")
print("==========================================")

display(
    year_feature_missing[
        year_feature_missing["missing_count"] > 0
    ].sort_values(
        ["feature", "year"]
    )
)


# ------------------------------------------------------------
# 11. MONTH-WISE MISSINGNESS
# ------------------------------------------------------------

month_feature_rows = []

for month, g in samples.groupby("month"):

    for feature in FEATURES:

        n_total = len(g)
        n_missing = int(g[feature].isna().sum())

        month_feature_rows.append({
            "month": month,
            "feature": feature,
            "total_records": n_total,
            "missing_count": n_missing,
            "missing_pct": (100.0 * n_missing / n_total) if n_total else np.nan,
        })

month_feature_missing = pd.DataFrame(month_feature_rows)

print("\n==========================================")
print("MONTH-WISE MISSINGNESS")
print("==========================================")

display(
    month_feature_missing[
        month_feature_missing["missing_count"] > 0
    ].sort_values(
        ["feature", "month"]
    )
)


# ------------------------------------------------------------
# 12. STATION COORDINATES
# ------------------------------------------------------------

station_coords = (
    samples.groupby("station_id")
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        records=("station_id", "size"),
    )
    .reset_index()
)

print("\n==========================================")
print("STATION COORDINATES")
print("==========================================")

display(station_coords)


# ------------------------------------------------------------
# 13. STATIC PREDICTOR STATUS
# ------------------------------------------------------------

print("\n==========================================")
print("STATIC PREDICTOR STATUS FROM NOTEBOOK 06")
print("==========================================")

display(static_qc)


# ------------------------------------------------------------
# 14. MODEL-COMBINATION COMPLETE-CASE COUNTS
# ------------------------------------------------------------

PRECIP8 = [
    "CCS",
    "PDIR",
    "GSMaP_MVK",
    "CDR",
    "CHIRPS",
    "IMERG",
    "GSMaP_Gauge",
    "ERA5",
]

LAND = [
    "DEM",
    "NDVI",
    "LST_Day",
    "Distance_Sea",
]

COMBINATIONS = {
    "Comb1": PRECIP8,
    "Comb1_land": PRECIP8 + LAND,
    "Comb2": ["CHIRPS", "CDR", "ERA5"],
    "Comb2_land": ["CHIRPS", "CDR", "ERA5"] + LAND,
}

split_years = {
    "train": [2017, 2018, 2019, 2020],
    "validation": [2021],
    "test": [2022],
}

complete_case_rows = []

for combo, features in COMBINATIONS.items():

    for split_name, years in split_years.items():

        d = samples[
            samples["year"].isin(years)
        ].copy()

        total = len(d)

        complete = d.dropna(
            subset=["rainfall_mm"] + features
        )

        n_complete = len(complete)

        complete_case_rows.append({
            "combination": combo,
            "split": split_name,
            "total_rows": total,
            "complete_rows": n_complete,
            "dropped_rows": total - n_complete,
            "complete_pct": (100.0 * n_complete / total) if total else np.nan,
            "stations_remaining": complete["station_id"].nunique(),
        })

complete_case_summary = pd.DataFrame(
    complete_case_rows
)

print("\n==========================================")
print("COMPLETE-CASE ROWS AVAILABLE FOR MODELLING")
print("==========================================")

display(complete_case_summary)


# ------------------------------------------------------------
# 15. STATION CONTRIBUTION AFTER COMPLETE-CASE FILTER
# ------------------------------------------------------------

station_contribution_rows = []

for combo, features in COMBINATIONS.items():

    for split_name, years in split_years.items():

        d = samples[
            samples["year"].isin(years)
        ].dropna(
            subset=["rainfall_mm"] + features
        )

        counts = (
            d.groupby("station_id")
            .size()
            .rename("rows")
            .reset_index()
        )

        counts["combination"] = combo
        counts["split"] = split_name

        station_contribution_rows.append(
            counts
        )

station_contribution = pd.concat(
    station_contribution_rows,
    ignore_index=True
)

print("\n==========================================")
print("STATION CONTRIBUTION AFTER COMPLETE-CASE FILTER")
print("==========================================")

display(
    station_contribution.sort_values(
        ["combination", "split", "station_id"]
    )
)


# ------------------------------------------------------------
# 16. SIMPLE DECISION FLAGS
# ------------------------------------------------------------

decision_rows = []

for feature in FEATURES:

    feature_df = station_feature_missing[
        station_feature_missing["feature"] == feature
    ]

    max_station_missing = feature_df["missing_pct"].max()
    stations_100_missing = int(
        (feature_df["missing_pct"] == 100.0).sum()
    )

    overall_pct = float(
        overall_missing.loc[feature, "missing_pct"]
    )

    if stations_100_missing > 0:
        flag = "REVIEW_SOURCE_COVERAGE"
    elif overall_pct > 20:
        flag = "HIGH_MISSING"
    elif overall_pct > 0:
        flag = "SOME_MISSING"
    else:
        flag = "OK"

    decision_rows.append({
        "feature": feature,
        "overall_missing_pct": overall_pct,
        "max_station_missing_pct": max_station_missing,
        "stations_100pct_missing": stations_100_missing,
        "flag": flag,
    })

decision_flags = pd.DataFrame(
    decision_rows
)

print("\n==========================================")
print("DECISION FLAGS")
print("==========================================")

display(decision_flags)


# ------------------------------------------------------------
# 17. SAVE ALL QC OUTPUTS
# ------------------------------------------------------------

OUT_DIR = PROCESSED_DIR / "station_coverage_qc"
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

overall_missing.to_csv(
    OUT_DIR / "overall_missing_by_feature.csv"
)

station_feature_missing.to_csv(
    OUT_DIR / "missing_by_station_feature.csv",
    index=False
)

station_missing_pivot.to_csv(
    OUT_DIR / "missing_count_pivot.csv"
)

status_by_station.to_csv(
    OUT_DIR / "status_by_station_feature.csv",
    index=False
)

year_feature_missing.to_csv(
    OUT_DIR / "missing_by_year_feature.csv",
    index=False
)

month_feature_missing.to_csv(
    OUT_DIR / "missing_by_month_feature.csv",
    index=False
)

station_coords.to_csv(
    OUT_DIR / "station_coordinates.csv",
    index=False
)

complete_case_summary.to_csv(
    OUT_DIR / "model_complete_case_summary.csv",
    index=False
)

station_contribution.to_csv(
    OUT_DIR / "station_contribution_after_filter.csv",
    index=False
)

decision_flags.to_csv(
    OUT_DIR / "decision_flags.csv",
    index=False
)


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n==========================================")
print("06B STATION COVERAGE QC COMPLETE")
print("==========================================")

print("QC folder:", OUT_DIR)

print("\nMost important table:")
print(OUT_DIR / "missing_by_station_feature.csv")

print("\nModel readiness table:")
print(OUT_DIR / "model_complete_case_summary.csv")

print("\nDecision flags:")
print(OUT_DIR / "decision_flags.csv")

print(
    "\nDo not proceed to final modelling until "
    "the 100%-missing station-feature pairs are reviewed."
)


PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh

Samples shape: (432, 19)
Stations: 6
Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


,station_id,year,month,date,latitude,longitude,rainfall_mm,DEM,Distance_Sea,CCS,PDIR,GSMaP_MVK,CDR,CHIRPS,IMERG,GSMaP_Gauge,ERA5,NDVI,LST_Day
0,CL503,2017,1,2017-01-01,22.6012,89.5195,0.0,5.0,32.322636,0.0,1.0,1.966016,NaN,4.258131,0.000,0.000000,0.208773,0.3041,23.28
1,CL503,2017,2,2017-02-01,22.6012,89.5195,0.0,5.0,32.322636,0.0,0.0,0.000000,NaN,5.859640,0.003,0.193933,3.796070,0.3495,27.74
2,CL503,2017,3,2017-03-01,22.6012,89.5195,345.0,5.0,32.322636,2.0,17.0,49.963665,NaN,47.325570,0.097,117.178435,101.051186,0.3115,29.21
3,CL503,2017,4,2017-04-01,22.6012,89.5195,270.0,5.0,32.322636,31.0,56.0,334.475193,NaN,93.311977,0.155,132.086874,85.262344,0.3005,31.95
4,CL503,2017,5,2017-05-01,22.6012,89.5195,783.0,5.0,32.322636,60.0,122.0,134.737306,NaN,149.536093,0.166,152.076353,107.824173,0.4503,32.39



OVERALL MISSINGNESS BY FEATURE


,missing_count,missing_pct
CCS,72,16.666667
PDIR,72,16.666667
CDR,72,16.666667
Distance_Sea,72,16.666667
LST_Day,47,10.879630
GSMaP_MVK,18,4.166667
CHIRPS,0,0.000000
IMERG,0,0.000000
ERA5,0,0.000000
GSMaP_Gauge,0,0.000000



MISSINGNESS BY STATION × FEATURE


...


STATION CONTRIBUTION AFTER COMPLETE-CASE FILTER


,station_id,rows,combination,split
8,CL504,12,Comb1,test
9,CL510,12,Comb1,test
10,CL515,11,Comb1,test
11,CL517,12,Comb1,test
0,CL504,48,Comb1,train
1,CL510,48,Comb1,train
2,CL515,44,Comb1,train
3,CL517,48,Comb1,train
4,CL504,12,Comb1,validation
5,CL510,12,Comb1,validation



DECISION FLAGS


,feature,overall_missing_pct,max_station_missing_pct,stations_100pct_missing,flag
0,CCS,16.666667,100.000000,1,REVIEW_SOURCE_COVERAGE
1,PDIR,16.666667,100.000000,1,REVIEW_SOURCE_COVERAGE
2,GSMaP_MVK,4.166667,8.333333,0,SOME_MISSING
3,CDR,16.666667,100.000000,1,REVIEW_SOURCE_COVERAGE
4,CHIRPS,0.000000,0.000000,0,OK
5,IMERG,0.000000,0.000000,0,OK
6,GSMaP_Gauge,0.000000,0.000000,0,OK
7,ERA5,0.000000,0.000000,0,OK
8,DEM,0.000000,0.000000,0,OK
9,NDVI,0.000000,0.000000,0,OK



06B STATION COVERAGE QC COMPLETE
QC folder: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_coverage_qc

Most important table:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_coverage_qc\missing_by_station_feature.csv

Model readiness table:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_coverage_qc\model_complete_case_summary.csv

Decision flags:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_coverage_qc\decision_flags.csv

Do not proceed to final modelling until the 100%-missing station-feature pairs are reviewed.
